# 🔍 Complete Exploratory Data Analysis (EDA) Master Walkthrough (Google Colab Notebook)
> **বাংলা টিউটোরিয়াল:** একটি বাস্তব টেলিযোগাযোগ চার্ন ডেটাসেট নিয়ে ধাপে ধাপে পূর্ণাঙ্গ এক্সপ্লোরেটরি ডেটা অ্যানালাইসিস।

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
print("Environment ready for Complete EDA!")

## ধাপ ১ ও ২: ডেটাসেট তৈরি ও প্রাথমিক স্ক্যান (Sanity Check)

In [ ]:
# কৃত্রিম কিন্তু বাস্তবমুখী ১০০০ জন কাস্টমারের ডেটাসেট
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'tenure_months': np.random.randint(1, 72, size=n),
    'monthly_charges': np.random.exponential(scale=30, size=n) + 20,
    'contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], size=n, p=[0.55, 0.25, 0.20]),
    'internet_service': np.random.choice(['Fiber optic', 'DSL', 'No'], size=n, p=[0.45, 0.35, 0.20]),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], size=n),
    'churn': np.random.choice([0, 1], size=n, p=[0.73, 0.27])
})

# কিছু ইচ্ছাকৃত মিসিং ও আউটলায়ার যোগ করা
df.loc[np.random.choice(n, 30), 'monthly_charges'] = np.nan
df.loc[np.random.choice(n, 15), 'contract'] = np.nan
df.loc[0, 'monthly_charges'] = 350.0 # চরম আউটলায়ার

print("Shape:", df.shape)
df.info()

## ধাপ ৩: মিসিং ডেটা বিশ্লেষণ

In [ ]:
missing_summary = pd.DataFrame({
    'Count': df.isna().sum(),
    'Percentage (%)': (df.isna().sum() / len(df)) * 100
})
print(missing_summary[missing_summary['Count'] > 0])

# ইম্পিউটেশন
df['monthly_charges'] = df['monthly_charges'].fillna(df['monthly_charges'].median())
df['contract'] = df['contract'].fillna(df['contract'].mode()[0])
print("ইম্পিউটেশনের পর মোট মিসিং:", df.isna().sum().sum())

## ধাপ ৪: একক চলক বিশ্লেষণ (Univariate: Distribution & Skewness)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# মাসিক খরচের ডিস্ট্রিবিউশন
sns.histplot(df['monthly_charges'], kde=True, ax=axes[0], color='teal')
axes[0].set_title(f"Monthly Charges\nSkewness: {df['monthly_charges'].skew():.2f}")

# টার্গেটের ব্যালেন্স চেক
sns.countplot(data=df, x='churn', ax=axes[1], palette='Set2')
churn_rate = df['churn'].mean() * 100
axes[1].set_title(f"Churn Distribution (Churn Rate: {churn_rate:.1f}%)")
plt.show()

## ধাপ ৫: দ্বি-চলক বিশ্লেষণ (Bivariate: Feature vs Target Churn)

In [ ]:
# ১. চুক্তি বনাম চার্ন (Chi-square Independence Test)
contract_churn = pd.crosstab(df['contract'], df['churn'], normalize='index') * 100
print("Contract vs Churn Percentage:\n", contract_churn)

chi2, p_val, _, _ = stats.chi2_contingency(pd.crosstab(df['contract'], df['churn']))
print(f"\nChi-Square Test p-value: {p_val:.4e}")
if p_val < 0.05:
    print(">> ইনসাইট: Contract এর ধরনের সাথে Churn হওয়ার শক্তিশালী সম্পর্ক রয়েছে!")

# ২. বক্সপ্লট: চার্ন করা গ্রাহকদের মেয়াদ (Tenure)
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='churn', y='tenure_months', palette='coolwarm')
plt.title("Tenure Months by Churn Status")
plt.show()

## ধাপ ৬: আউটলায়ার হ্যান্ডলিং (IQR Capping)

In [ ]:
q1 = df['monthly_charges'].quantile(0.25)
q3 = df['monthly_charges'].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr
lower_limit = q1 - 1.5 * iqr

outliers_count = ((df['monthly_charges'] > upper_limit) | (df['monthly_charges'] < lower_limit)).sum()
print(f"আউটলায়ার সংখ্যা: {outliers_count}")

# ক্যাপিং বা উইনসোরাইজেশন (Winsorization)
df['monthly_charges'] = np.clip(df['monthly_charges'], lower_limit, upper_limit)
print("ক্যাপিং সম্পন্ন!")